# Lab 1 - Exploring Table Data

You are to perform preprocessing and exploratory analysis of a data set: exploring the statistical summaries of the features, visualizing the attributes, and addressing data quality. This report is worth 10% of the final grade. Please upload a report (one per team) with all code used, visualizations, and text in a rendered Jupyter notebook. Any visualizations that cannot be embedded in the notebook, please provide screenshots of the output. Only the rendered HTML notebook is acceptable. 

------

Prediction: Will a telecom customer churn (Yes/No)?

Questions
- How does contract type (month-to-month vs yearly) affect churn rate?
- Do customers with certain InternetService types (DSL, Fiber, None) churn more?
- What is the relationship between tenure and monthly charges for churners vs non-churners?
- Are add-on services like TechSupport linked to lower churn?
- Does payment method (auto-pay vs manual) influence churn likelihood?

------

## 1. Business Understanding

### 1.1 Business Understanding

The main question we hope to answer with our analysis is whether a Telco customer will churn (discontinue their service with a provider). Churn is one of the most important challenges in the telecommunications industry, as customer acquisition costs are high and the industry is highly competitive. Retaining a new customer is estimated to be from 5 to 25 times cheaper than acquiring a new one, due to the lower marketing, sales, and onboarding costs, meaning that making an effective churn prediction can greatly impact the profitability of a company. As the dataset description says: “The data set includes information about:
- Customers who left within the last month – the column is called Churn
- Services that each customer has signed up for – phone, multiple lines, internet, online security, online backup, device protection, tech support, and streaming TV and movies
- Customer account information – how long they’ve been a customer, contract, payment method, paperless billing, monthly charges, and total charges
- Demographic info about customers – gender, age range, and if they have partners and dependents”

Analyzing these features will allow us to determine which factors most strongly influence customer churn or retention.

The specific business questions we are asking include:
- Contract type: How does contract type (month-to-month vs yearly) affect churn rate.
- Internet service type: Do customers with certain Internet Service types (DSL, Fiber, None) churn more?
- Tenure vs. Charges: What is the relationship between tenure and monthly charges for churners vs. non-churners?
- Value-added Services: Are add-on services like tech support linked to lower churn?
- Payment Method: Does payment method (auto-pay vs manual) influence churn likelihood?     

The end goal of analyzing this dataset would be to classify whether a customer is likely to churn, while also providing actionable insights. 

Beyond just the prediction, the analysis can guide business strategy, such as:
- Encouraging customers to switch to more long-term contracts
- Bundling internet and support services
- Offering incentives to customers with higher monthly charges
- Promoting auto-pay enrollment to lower attrition rates

### 1.2 Measure of Success

Success in this context is more than just base accuracy. Because churn prediction is either “yes or no”, a binary classification problem, a model that always predicts “no churn” could already be at a 70% accuracy, but it would fail to identify at-risk customers. Therefore, we should focus on:
- Recall for churners (minimizing false negatives): Making sure that we correctly flag as many at-risk customers as possible, since losing them directly impacts revenue
- Precision for churners (minimizing false positives): Avoiding the waste of resources for customers who were already unlikely to leave

A churn model would be considered viable for a business if it identifies a majority of churners (>80-85% recall) while keeping false positives at a low number, so that the cost of interventions is outweighed by the value of the customers retained.

Dataset Source: [https://www.kaggle.com/datasets/blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

------

## 2. Data Understanding

### 2.1 Data Description

In [32]:
import warnings
warnings.filterwarnings("ignore")

In [28]:
import pandas as pd
import numpy as np

df = pd.read_csv('/workspace/datasets/Lab1.csv')

df = df.rename(columns={"customerID": "CustomerId", "gender": "Gender", "tenure": "Tenure"})

df.head()

,CustomerId,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


| Column | Old Data Type | New Data Type |
| ------ | ------------- | ------------- |
| Gender | String (Male/Female) | Bool (Male = 0, Female = 1) |
| Partner | String (Yes, No) | Bool (yes = 1, no = 0) |
| Dependents | String (Yes, No) | Bool (yes = 1, no = 0) |
| PhoneService | String (Yes, No) | Bool (yes = 1, no = 0) |
| MultipleLines | String (Yes, No, No phone service) | Int (yes = 1, no = 2, nps = 3) |
| InternetService | String (DSL, Fiber optic, No) | Int (DSL = 1, Fiber optic = 2, no = 3) |
| OnlineSecurity | String (Yes, No, No internet service) | Int (yes = 1, no = 2, nis = 3) |
| OnlineBackup | String (Yes, No, No internet service) | Int (yes = 1, no = 2, nis = 3) |
| DeviceProtection | String (Yes, No, No internet service) | Int (yes = 1, no = 2, nis = 3) |
| TechSupport | String (Yes, No, No internet service) | Int (yes = 1, no = 2, nis = 3) |
| StreamingTV | String (Yes, No, No internet service) | Int (yes = 1, no = 2, nis = 3) |
| StreamingMovies | String (Yes, No, No internet service) | Int (yes = 1, no = 2, nis = 3) |
| Contract | String (Month-to-month, One year, Two year) | Int (mtm = 1, oy = 2, ty = 3) |
| PaperlessBilling | String (Yes, No) | Bool (yes = 1, no = 0) |
| PaymentMethod | String (Electronic check, Mailed check, Bank transfer (automatic), Credit card (automatic)) | Int (ec = 1, mc = 2, bs = 3, cc = 4) |
| Churn | String (Yes, No) | Bool (yes = 1, no = 0) |

In [38]:
df.Gender.replace(to_replace = ['Male', 'Female'], value = range(0,2), inplace = True)
df.Partner.replace(to_replace = ['No', 'Yes'], value = range(0,2), inplace = True)
df.Dependents.replace(to_replace = ['No', 'Yes'], value = range(0,2), inplace = True)
df.PhoneService.replace(to_replace = ['No', 'Yes'], value = range(0,2), inplace = True)
df.MultipleLines.replace(to_replace = ['Yes', 'No', 'No phone service'], value = range(1,4), inplace = True)
df.InternetService.replace(to_replace = ['DSL', 'Fiber optic', 'No'], value = range(1,4), inplace = True)
df.OnlineSecurity.replace(to_replace = ['Yes', 'No', 'No internet service'], value = range(1,4), inplace = True)
df.OnlineBackup.replace(to_replace = ['Yes', 'No', 'No internet service'], value = range(1,4), inplace = True)
df.DeviceProtection.replace(to_replace = ['Yes', 'No', 'No internet service'], value = range(1,4), inplace = True)
df.TechSupport.replace(to_replace = ['Yes', 'No', 'No internet service'], value = range(1,4), inplace = True)
df.StreamingTV.replace(to_replace = ['Yes', 'No', 'No internet service'], value = range(1,4), inplace = True)
df.StreamingMovies.replace(to_replace = ['Yes', 'No', 'No internet service'], value = range(1,4), inplace = True)
df.Contract.replace(to_replace = ['Month-to-month', 'One year', 'Two year'], value = range(1,4), inplace = True)
df.PaperlessBilling.replace(to_replace = ['No', 'Yes'], value = range(0,2), inplace = True)
df.PaymentMethod.replace(to_replace = ['Electronic check', 'Mailed check', 'Bank transfer (automatic)', 'Credit card (automatic)'], value = range(1,5), inplace = True)
df.Churn.replace(to_replace = ['No', 'Yes'], value = range(0,2), inplace = True)

df.head()

,CustomerId,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,1,0,1,0,1,0,3,1,2,...,2,2,2,2,1,1,1,29.85,29.85,0
1,5575-GNVDE,0,0,0,0,34,1,2,1,1,...,1,2,2,2,2,0,2,56.95,1889.5,0
2,3668-QPYBK,0,0,0,0,2,1,2,1,1,...,2,2,2,2,1,1,2,53.85,108.15,1
3,7795-CFOCW,0,0,0,0,45,0,3,1,1,...,1,1,2,2,2,0,3,42.30,1840.75,0
4,9237-HQITU,1,0,0,0,2,1,2,2,2,...,2,2,2,2,1,1,1,70.70,151.65,1
